In [24]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set seed for reproducibility
np.random.seed(42)

# Using Path().resolve() for Notebook consistency (mimics __file__ for interactive environments)
# Traverse up two levels from training/mcmc to project root



PROJECT_ROOT = Path().resolve().parent.parent

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

MOCK_CSV_PATH = DATA_DIR / "mock_drill_logs.csv"

In [ ]:
n_boreholes = 25
data = {
    "borehole_id": [f"BH-{i+1:03d}" for i in range(n_boreholes)],
    "depth_m": np.random.uniform(10, 150, n_boreholes).round(1),
    "rmr_score": np.clip(np.random.normal(52, 12, n_boreholes), 0, 100).round(1),
    "rock_class": np.random.choice(["II", "III", "IV", "V"], n_boreholes, p=[0.1, 0.4, 0.35, 0.15]),
    "ucs_mpa": np.clip(np.random.lognormal(3.8, 0.4, n_boreholes), 5, 200).round(1),
    "rqd_pct": np.clip(np.random.beta(5, 3, n_boreholes) * 100, 0, 100).round(1),
}

# Inject some NaN values (missing data — reality!)
data["ucs_mpa"][3] = np.nan
data["ucs_mpa"][17] = np.nan
data["rqd_pct"][8] = np.nan

df = pd.DataFrame(data)

# Save to the defined path
df.to_csv(MOCK_CSV_PATH, index=False)
#print(f"Borehole data orchestrated to: {MOCK_CSV_PATH}")

df.head()

In [ ]:
class DrillLogPipeline: # lets keep it super specific to our csv
    """Ingests, validates, and formats drill-log data for PyMC models."""

    def __init__(self, filepath: Path ) -> None:
        self.filepath = filepath
        self.raw_data = None
        self.clean_data = None


    def load(self) -> pd.DataFrame:
        """load the raw csv"""
        self.raw_data = pd.read_csv(self.filepath)
        print(f"Loaded {len(self.raw_data)} rows from {self.filepath}")
        return self.raw_data

    def check_null(self):
        """High-signal Forensic Audit: only reports on Data Voids (NaNs)."""
        null_counts = self.raw_data.isnull().sum()
        # Filter for only parameters containing voids
        self.data_voids = null_counts[null_counts > 0]

        if self.data_voids.empty:
            print("Forensic Audit: No data voids detected.")
        else:
            print("Forensic Alert: Data voids found in high-priority parameters:")
            for param, count in self.data_voids.items():
                print(f"  - {param}: {count} null values")
        return self.data_voids

    def check_range(self, column_targets: list, min_val: float = 0, max_val: float = 100):
        """
        Checks multiple columns against a specified physical range.
        Handles both column names (str) and indices (int).
        """
        results = {}

        for target in column_targets:
            # Map index to name if integer provided
            col_name = self.raw_data.columns[target] if isinstance(target, int) else target

            # Vectorized Range Audit
            out_of_range = np.sum((self.raw_data[col_name] < min_val) | (self.raw_data[col_name] > max_val)).item() # in print 0 instead of np.int64(0)
            results[col_name] = out_of_range

            if out_of_range > 0:
                print(f"Forensic Alert: [{col_name}] has {out_of_range} values outside [{min_val}, {max_val}]")

        return results

In [ ]:
pipeline = DrillLogPipeline(MOCK_CSV_PATH)

pipeline.load()
pipeline.check_null()
pipeline.check_range([2, 4,5])
